# MedScribe — Kişi 3: SOAP notu üretimi (Colab)

**Ne yapar:** ACI-Bench'ten dönüştürülmüş 3 diyaloğun her biri için 2 not üretir — **rollü** (`gold_role`) ve **rolsüz** (`None`) — toplam 6 not (plan Bölüm 3).

**Çalıştırmadan önce:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`.

**Süre:** ilk seferde model indirme ~4 dk + 6 not ~4 dk.

**Girdi:** GitHub reposu (aşağıda klonlanır). İstem `notes/istem_v1.txt` içinde **dondurulmuştur**; kodda istem kopyası yoktur.

**Çıktı:** `notes/outputs/aci_00X_rollu.json`, `..._rolsuz.json`, `karsilastirma.md`. Bunlar repoya commit edilir (Salı teslimi).

In [ ]:
# 1) Donanım
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU yok: Çalışma zamanı → T4 GPU seçin'

In [ ]:
# 2) Bağımlılıklar (~1 dk)
!pip -q install -U transformers accelerate bitsandbytes

In [ ]:
# 3) Repoyu al
REPO = "https://github.com/Pikseel/medscribe.git"
PROJECT = "/content/medscribe"

import os, sys
if not os.path.isdir(PROJECT):
    !git clone -q $REPO $PROJECT
%cd $PROJECT
!git pull -q
sys.path.insert(0, PROJECT)
print(sorted(os.listdir('.')))

In [ ]:
# 4) Veri formatı kontrolü (common/format.py)
!python -m common.format notes/data/aci_001.json notes/data/aci_002.json notes/data/aci_003.json

In [ ]:
# 5) Modeli yükle: 4-bit Qwen2.5-7B-Instruct (T4'e sığar)
import notes.generate_note as gn
gn.set_backend('transformers')
gn._load_hf()
print('model hazır:', gn.model_name(), '| backend:', gn.BACKEND)

In [ ]:
# 6) İstem testi: plan Bölüm 2.5b örnek diyaloğu
import json
d = json.load(open('common/ornek_diyalog.json'))
note = gn.generate_note(d, 'gold_role')
print(json.dumps(note, ensure_ascii=False, indent=2))
# Beklenen: bir haftadır öksürük, gece hafif ateş, GÖĞÜS AĞRISI YOK.
# 'chest pain' olumlu yazıldıysa istem zayıf demektir -> istem_v2 gerekir, DURUN.

In [ ]:
# 7) Deney: 3 diyalog x 2 koşul = 6 not
import time
rows = []
for path, rf, cond in gn.EXPERIMENT:
    d = json.load(open(path))
    t0 = time.time()
    note = gn.generate_note(d, rf)
    out = f"notes/outputs/{d['dialogue_id']}_{cond}.json"
    gn.save_note(note, out)
    rows.append((d['dialogue_id'], cond, len(note['S']), len(note['O']),
                 len(note['A']), len(note['P']), round(time.time() - t0)))
    print(f"{out:44s} {rows[-1][6]:3d}s")

print()
print(f"{'diyalog':10s} {'koşul':8s} {'S':>5s} {'O':>5s} {'A':>5s} {'P':>5s}")
for r in rows:
    print(f"{r[0]:10s} {r[1]:8s} {r[2]:5d} {r[3]:5d} {r[4]:5d} {r[5]:5d}")

In [ ]:
# 8) Rollü vs rolsüz karşılaştırma + olumsuzlama bayrakları
!python -m notes.compare_notes
from IPython.display import Markdown, display
display(Markdown(open('notes/outputs/karsilastirma.md', encoding='utf-8').read()))

In [ ]:
# 9) Sonuçları indir (+ istersen Drive'a kopyala)
!cd /content && zip -q -r kisi3_notlar.zip medscribe/notes/outputs
from google.colab import files
files.download('/content/kisi3_notlar.zip')

SAVE_TO_DRIVE = False   # True yaparsan MedScribe_Drive/notes_outputs altına kopyalar
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/MedScribe_Drive/notes_outputs
    !cp notes/outputs/*.json notes/outputs/karsilastirma.md /content/drive/MyDrive/MedScribe_Drive/notes_outputs/